# 02 — Data Preprocessing & Sliding-Window Segmentation

Cleans raw sensor readings, maps the target activity codes to 6 classes (`Walking`, `Jogging`, `Stairs`, `Sitting`, `Standing`, `Eating`), creates sliding windows without crossing subject boundaries, splits into train/val/test, and standardizes using training-set statistics only.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import load_config
from src.data import (
    ChannelStandardizer,
    clean_sensor_dataframe,
    create_segments,
    filter_and_map_activities,
    load_wisdm_raw,
    save_processed_splits,
    stratified_train_val_test_split,
)

cfg = load_config(PROJECT_ROOT / "configs" / "config.yaml")
data_cfg = cfg["data"]
seg_cfg = cfg["segmentation"]
split_cfg = cfg["split"]

raw_df = load_wisdm_raw(
    raw_dir=data_cfg["raw_dir"],
    devices=data_cfg["devices"],
    sensors=data_cfg["sensors"],
    fallback_dir=PROJECT_ROOT / data_cfg["fallback_raw_dir"],
)
cleaned_df = clean_sensor_dataframe(raw_df, features=data_cfg["features"])
filtered_df = filter_and_map_activities(cleaned_df, activity_map=data_cfg["activity_map"])
print("Filtered records:", len(filtered_df))

In [ ]:
X, y = create_segments(
    filtered_df,
    time_steps=int(seg_cfg["time_steps"]),
    step=int(seg_cfg["step"]),
    features=data_cfg["features"],
    label_strategy=str(seg_cfg.get("label_strategy", "majority")),
)
splits = stratified_train_val_test_split(
    X, y,
    test_size=float(split_cfg["test_size"]),
    val_size=float(split_cfg["val_size"]),
    random_state=int(split_cfg["random_state"]),
)
scaler = ChannelStandardizer(features=data_cfg["features"])
splits["X_train"] = scaler.fit_transform(splits["X_train"])
splits["X_val"] = scaler.transform(splits["X_val"])
splits["X_test"] = scaler.transform(splits["X_test"])

scaler.save(PROJECT_ROOT / cfg["training"]["scaler_path"])
save_processed_splits(splits, PROJECT_ROOT / data_cfg["processed_dir"])
print({k: v.shape for k, v in splits.items()})